---
## 1. Importaciones y carga de textos

In [ ]:
import random

# Los textos ya fueron preprocesados externamente:
#   - sin puntuación
#   - sin mayúsculas
#   - sin acentos
# Esto simplifica el vocabulario y evita que "casa", "Casa" y "casa," se traten como palabras distintas.

with open("EL_PRINCIPITO_limpio.txt", "r", encoding="utf-8") as archivo:
    principito = archivo.read().split()   # Lista de palabras

with open("No_Tengo_Boca_Y_Debo_Gritar_limpio.txt", "r", encoding="utf-8") as archivo:
    ntbydg = archivo.read().split()       # Lista de palabras

print(f"El Principito         → {len(principito):,} palabras")
print(f"No Tengo Boca...      → {len(ntbydg):,} palabras")
print()
print("Primeras 10 palabras de El Principito:", principito[:10])

---
## 2. Construcción de las tablas de frecuencias


In [ ]:
def tabla_frecuencias(texto):
    """
    Construye una tabla de frecuencias de orden 1.
    
    Para cada par de palabras consecutivas (wᵢ, wᵢ₊₁) en el texto,
    cuenta cuántas veces aparece ese par.

    """
    tabla = {}
    for i in range(len(texto) - 1):
        par = (texto[i], texto[i + 1])
        tabla[par] = tabla.get(par, 0) + 1
    return tabla


def tabla_frecuencias_orden2(texto):
    """
    Construye una tabla de frecuencias de orden 2.

    Para cada tripleta de palabras consecutivas (wᵢ, wᵢ₊₁, wᵢ₊₂),
    cuenta cuántas veces aparece esa tripleta.
    Permite que la predicción dependa de las dos últimas palabras,
    generando textos más coherentes que el orden 1.

    """
    tabla = {}
    for i in range(len(texto) - 2):
        tripleta = (texto[i], texto[i + 1], texto[i + 2])
        tabla[tripleta] = tabla.get(tripleta, 0) + 1
    return tabla


tabla1_principito = tabla_frecuencias(principito)
tabla1_ntbydg     = tabla_frecuencias(ntbydg)

tabla2_principito = tabla_frecuencias_orden2(principito)
tabla2_ntbydg     = tabla_frecuencias_orden2(ntbydg)

print(f"Pares únicos en El Principito (orden 1):         {len(tabla1_principito):,}")
print(f"Tripletas únicas en El Principito (orden 2):     {len(tabla2_principito):,}")
print(f"Pares únicos en No Tengo Boca... (orden 1):      {len(tabla1_ntbydg):,}")
print(f"Tripletas únicas en No Tengo Boca... (orden 2):  {len(tabla2_ntbydg):,}")

**Pares frecuentes de *El Principito***

In [ ]:
# Top 5 pares más frecuentes en El Principito
top5 = sorted(tabla1_principito.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 pares (El Principito):")
for par, freq in top5:
    print(f"  {par[0]:>15} → {par[1]:<15}  veces: {freq}")

---
## 3. Función de predicción

Dada la secuencia generada hasta el momento, esta función calcula una **distribución de probabilidad** sobre las posibles siguientes palabras.

1. Si hay al menos 2 palabras de contexto, busca en la tabla de **orden 2**.
2. Si no hay datos suficientes en orden 2, cae a la tabla de **orden 1**.
3. Las frecuencias se normalizan para obtener probabilidades que sumen 1.

In [ ]:
def predecir(contexto, tabla1, tabla2):
    """
    Calcula la distribución de probabilidad de la siguiente palabra.

    Prioriza el contexto de orden 2 (últimas 2 palabras); si no hay
    datos, recurre al orden 1 (última palabra).

    """
    if len(contexto) >= 2:
        a, b = contexto[-2], contexto[-1]
        # Filtramos tripletas cuyas dos primeras palabras coincidan con el contexto
        conteos = {c: cnt for (x, y, c), cnt in tabla2.items() if (x, y) == (a, b)}
        if conteos:
            total = sum(conteos.values())
            return {pal: cnt / total for pal, cnt in conteos.items()}

    a = contexto[-1]
    conteos = {b: cnt for (x, b), cnt in tabla1.items() if x == a}
    if not conteos:
        return {}  # Palabra nunca vista como inicio de par
    total = sum(conteos.values())
    return {pal: cnt / total for pal, cnt in conteos.items()}


# Ejemplo de uso
dist_ejemplo = predecir(["el", "principito"], tabla1_principito, tabla2_principito)
top_candidatas = sorted(dist_ejemplo.items(), key=lambda x: x[1], reverse=True)[:8]

print("Palabras más probables después de 'el principito':")
for palabra, prob in top_candidatas:
    barra = "█" * int(prob * 40)
    print(f"  {palabra:<15} {prob:.3f}  {barra}")

---
## 4. Temperatura

In [ ]:
def ajustar_temperatura(distribucion, temperatura):
    """
    Redistribuye las probabilidades según el parámetro de temperatura.
    """
    if temperatura <= 0:
        raise ValueError("La temperatura debe ser mayor que 0.")
    ajustada = {pal: prob ** (1.0 / temperatura) for pal, prob in distribucion.items()}
    total = sum(ajustada.values())
    return {pal: prob / total for pal, prob in ajustada.items()}


# Efecto de la temperatura sobre la misma distribución
dist_base = predecir(["le", "dije"], tabla1_principito, tabla2_principito)
if not dist_base:
    dist_base = predecir(["dije"], tabla1_principito, tabla2_principito)

candidatas = sorted(dist_base.items(), key=lambda x: x[1], reverse=True)[:6]
palabras = [p for p, _ in candidatas]

print(f"{'Palabra':<15}  {'T=0.5':>8}  {'T=1.0':>8}  {'T=1.5':>8}")
print("-" * 45)
for temp in [0.5, 1.0, 1.5]:
    dist_t = ajustar_temperatura(dict(candidatas), temp)

if candidatas:
    rows = {p: {} for p, _ in candidatas}
    for temp in [0.5, 1.0, 1.5]:
        dist_t = ajustar_temperatura(dict(candidatas), temp)
        for p in rows:
            rows[p][temp] = dist_t.get(p, 0)
    print(f"{'Palabra':<15}  {'T=0.5':>8}  {'T=1.0':>8}  {'T=1.5':>8}")
    print("-" * 45)
    for p, temps in rows.items():
        print(f"{p:<15}  {temps[0.5]:>8.4f}  {temps[1.0]:>8.4f}  {temps[1.5]:>8.4f}")

---
## 5. Generación de texto

Generando texto de longitud dada partiendo de palabra aleatoria

In [ ]:
def generar_texto(tabla1, tabla2, texto_original, n_palabras=50, temperatura=1.0):
    """
    Genera una secuencia de palabras usando la cadena de Markov.
    """
    # Palabra inicial: elegida al azar del vocabulario
    palabra_inicial = random.choice(list({a for (a, _) in tabla1.keys()}))
    generado = [palabra_inicial]

    for _ in range(n_palabras - 1):
        distribucion = predecir(generado, tabla1, tabla2)

        if not distribucion:
            # Sin salida posible: reiniciar contexto
            generado.append(random.choice(texto_original))
            continue

        distribucion = ajustar_temperatura(distribucion, temperatura)

        # Muestreo ponderado por probabilidad
        siguiente = random.choices(
            list(distribucion.keys()),
            weights=distribucion.values()
        )[0]
        generado.append(siguiente)

    return " ".join(generado)


# Ejemplo
print("Texto de prueba (El Principito, 30 palabras, T=1.0):")
print(generar_texto(tabla1_principito, tabla2_principito, principito, n_palabras=30))

---
## 6. Comparación entre corpus

Generamos 50 palabras con cada corpus a temperatura normal (T = 1.0) 

In [ ]:
def comparar_textos(texto1, texto2, etiqueta1="Corpus 1", etiqueta2="Corpus 2"):
    """
    Imprime los dos textos generados uno al lado del otro para comparar
    cualitativamente el efecto del corpus en el estilo del texto producido.
    """
    print(f"\n{'─'*55}")
    print(f"Texto generado – {etiqueta1}:")
    print(texto1)
    print(f"\nTexto generado – {etiqueta2}:")
    print(texto2)
    print(f"{'─'*55}")


print("=" * 55)
print("  GENERACIÓN DE TEXTO – temperatura = 1.0 (normal)")
print("=" * 55)

texto_principito = generar_texto(
    tabla1_principito, tabla2_principito, principito, n_palabras=50, temperatura=1.0
)
texto_ntbydg = generar_texto(
    tabla1_ntbydg, tabla2_ntbydg, ntbydg, n_palabras=50, temperatura=1.0
)

comparar_textos(
    texto_principito, texto_ntbydg,
    etiqueta1="El Principito",
    etiqueta2="No Tengo Boca y Debo Gritar"
)

---
## 7. Efecto de la temperatura

Generamos el mismo tipo de texto con tres temperaturas distintas (0.5, 1.0, 1.5) 

In [ ]:
print("=" * 55)
print("  EFECTO DE TEMPERATURA  (El Principito, 50 palabras)")
print("=" * 55)

for temp in [0.5, 1.0, 1.5]:
    print(f"\n  Temperatura = {temp}:")
    print(generar_texto(
        tabla1_principito, tabla2_principito, principito,
        n_palabras=50, temperatura=temp
    ))

In [ ]:
print("=" * 69)
print("  EFECTO DE TEMPERATURA  (No tengo boca y debo gritar, 50 palabras)")
print("=" * 69)

for temp in [0.5, 1.0, 1.5]:
    print(f"\n  Temperatura = {temp}:")
    print(generar_texto(
        tabla1_ntbydg, tabla2_ntbydg, ntbydg,
        n_palabras=50, temperatura=temp
    ))

---
## 8. Análisis de transiciones más frecuentes

In [ ]:
def transiciones_frecuentes(tabla, n=10):
    """
    Imprime las n transiciones (pares o tripletas) más frecuentes de una tabla.

    """
    transiciones = sorted(tabla.items(), key=lambda item: item[1], reverse=True)[:n]
    for transicion, frecuencia in transiciones:
        print(f"  {str(transicion):<45}  →  {frecuencia}")


print("=" * 55)
print("  TRANSICIONES MÁS FRECUENTES (orden 1)")
print("=" * 55)

print("\nEl Principito:")
transiciones_frecuentes(tabla1_principito)

print("\nNo Tengo Boca y Debo Gritar:")
transiciones_frecuentes(tabla1_ntbydg)

In [ ]:
print("=" * 55)
print("  TRANSICIONES MÁS FRECUENTES (orden 2)")
print("=" * 55)

print("\nEl Principito:")
transiciones_frecuentes(tabla2_principito)

print("\nNo Tengo Boca y Debo Gritar:")
transiciones_frecuentes(tabla2_ntbydg)

---
## 9. Reflexión final

### Limitaciones del modelo
| Limitación | Descripción |
|---|---|
| Sin memoria larga | Solo recuerda las últimas 1-2 palabras, no el tema global del texto |
| Sin gramática | No garantiza oraciones gramaticalmente correctas |
| Sin semántica | No "comprende" el significado, solo las co-ocurrencias |
| Dependiente del tamaño del corpus | Con poco texto, el vocabulario y las transiciones posibles son muy limitados |
